In [ ]:
import os
import pandas as pd
import sys

project_root = os.path.abspath("..")   # lên 1 cấp: MIND-research

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [ ]:
project_root

In [ ]:
from src.core.Context import VectorContext
from src.urv.URV import URV
from src.recommendation.RecommendationEngine import RecommendationEngine
from src.represent.RepresentedVector import RepresentedVector
from src.matrix.Matrix import Metrix
import numpy as np

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from src.matrix.Matrix import Metrix

In [ ]:
# Semantic pretrained
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")

# Topic model đã train
topic_model = BERTopic.load(project_root + "/models/primary/bertopic")

In [ ]:
impressions_path = os.path.join(
    project_root,
    "data",
    "primary",
    "dev_set",
    "behaviors.tsv"
)

columns_impressions = [
    "behavior_id",
    "user_id",
    "time",
    "history",
    "impressions",
]

impressions_df = pd.read_csv(
    impressions_path,
    sep="\t",
    names=columns_impressions,
)

impressions_df = impressions_df[
    ["behavior_id", "user_id", "history", "impressions"]
]

impressions_df = impressions_df.dropna(subset=["history"])
impressions_df = impressions_df.dropna(subset=["impressions"])

impressions_df = impressions_df.to_dict(orient="records")

In [ ]:
context = VectorContext(os.path.join(project_root, "data", "primary", "dev_set"))
display(len(impressions_df))

In [ ]:
title_list = context.createTitleList()
title_list = {item["news_id"]: item["title"] for item in title_list}

represented_vector = RepresentedVector(semantic_model=semantic_model, topic_model=topic_model)
urv = URV()
recommender = RecommendationEngine()

In [ ]:
results = []

for row in impressions_df:
    sample = context.createImpressionRow(row)
    history_title_vector = {news_id: represented_vector.get_vector(title_list[news_id])
                            for news_id in sample["history"].split(" ")}
    
    sample_urv = urv.getURVFromVector(history_title_vector.values())
    cadidate_list = [{"id": i["news_id"],
                    "vector": represented_vector.get_vector(title_list[i["news_id"]]),
                    "label": i["label"]}
                    for i in sample["impressions"]]
    impression = recommender.recommend(sample_urv, cadidate_list)
    results.append(Metrix(impression).evaluate())
    
print("AUC:", np.mean([r["AUC"] for r in results]))
print("MRR:", np.mean([r["MRR"] for r in results]))
print("nDCG@5:", np.mean([r["nDCG@5"] for r in results]))
print("nDCG@10:", np.mean([r["nDCG@10"] for r in results]))
    
    